# AwareLiquid · Track 1B canonical — real-inference trace with Phase 5b adapter

Runs `scripts/awareliquid_real_trace.py` against Qwen-2.5-1.5B-Instruct with the Phase 5b MT residual adapter loaded. Produces the canonical adapter-on real `*.trace.jsonl` artifact (entropy-thresholded LOCAL/SELF_CRITIQUE/CLOUD routing + `[Absorbed fact]` splicing).

Mounts Phase 5b output as input dataset `muningan/awareliquid-phase-5b`. Wall: ~5–15 min on T4.

## 0 · Pin torch for sm_60 + sm_75

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    'torch==2.4.1', 'torchvision==0.19.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'])
print('torch pinned to 2.4.1+cu121')
import sys as _s
if 'torch' in _s.modules:
    import os, signal; os.kill(os.getpid(), signal.SIGTERM)

## 1 · Clone M1 + GPU sanity

In [ ]:
import os, subprocess, torch
REPO, DIR = 'https://github.com/everest-an/M1.git', '/kaggle/working/M1'
if not os.path.exists(DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, DIR])
os.chdir(DIR); subprocess.check_call(['git', 'log', '-1', '--oneline'])
cap = torch.cuda.get_device_capability(0)
print('gpu:', torch.cuda.get_device_name(0), f'sm_{cap[0]}{cap[1]}')

In [ ]:
!pip install -q -r requirements.txt accelerate safetensors peft datasets

## 2 · Locate Phase 5b adapter checkpoint

In [ ]:
import glob, os
candidates = sorted(glob.glob('/kaggle/input/**/llama_mt_adapter_001000.pt', recursive=True))
print('checkpoints found:')
for c in candidates: print(' ', c, os.path.getsize(c)//1024, 'KB')
assert candidates, 'no Phase 5b adapter checkpoint mounted'
ADAPTER = candidates[-1]
os.environ['ADAPTER'] = ADAPTER
print('using:', ADAPTER)

## 3 · Run real-inference trace (3 prompts × adapter-on)

In [ ]:
import os
os.makedirs('/kaggle/working/traces', exist_ok=True)
prompts = [
    ('canberra', 'The capital of Australia is', 'Canberra is the capital city of Australia, located in the ACT.'),
    ('einstein', 'The theory of general relativity was published by', 'Albert Einstein published the general theory of relativity in 1915.'),
    ('mtheory',  'The origin of M-theory was proposed by',         'Edward Witten proposed M-theory in 1995, unifying five string theories.'),
]
for name, prompt, fact in prompts:
    out = f'/kaggle/working/traces/qwen15b_adapter_{name}.trace.jsonl'
    cmd = (
        f'PYTHONPATH=/kaggle/working/M1 python scripts/awareliquid_real_trace.py '
        f'--model Qwen/Qwen2.5-1.5B-Instruct --adapter {os.environ["ADAPTER"]} '
        f'--prompt "{prompt}" --cloud_fact "{fact}" '
        f'--out {out} --max_new 80 --temperature 0.7 --session_id qwen15b_adapter_{name}'
    )
    print('---', name, '---')
    print(os.popen(cmd).read())

## 4 · Audit each trace

In [ ]:
import glob, subprocess
traces = sorted(glob.glob('/kaggle/working/traces/*.trace.jsonl'))
print('traces:', traces)
for t in traces:
    print('===', t, '===')
    print(subprocess.check_output(['python', '/kaggle/working/M1/scripts/bench_trace_audit.py', t]).decode())

## 5 · Save JSON audits + bundle

In [ ]:
import glob, subprocess, shutil
from pathlib import Path
out = Path('/kaggle/working/real_trace_artifacts'); out.mkdir(exist_ok=True)
for t in sorted(glob.glob('/kaggle/working/traces/*.trace.jsonl')):
    shutil.copy(t, out / Path(t).name)
    aj = (out / Path(t).name).with_suffix('.audit.json')
    aj.write_text(subprocess.check_output(['python', '/kaggle/working/M1/scripts/bench_trace_audit.py', '--format', 'json', t]).decode())
archive = shutil.make_archive('/kaggle/working/real_trace_qwen15b_adapter', 'zip', out)
print('archive:', archive)